In [13]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import pickle

from collections import Counter
    
with open('commit_counts_by_repo.pkl', 'rb') as f:
    commit_count_repos = pickle.load(f)

df = pd.read_csv('CycloneNSpdxTools_with_dates.csv')

spdx_counter = Counter()
cyclonedx_counter = Counter()
for index, row in df.iterrows():
    if row['Format'] == 'SPDX':
        spdx_counter.update(v for v in commit_count_repos[row['Repo']].values() if v > 1)
    elif row['Format'] == 'CycloneDx':
        cyclonedx_counter.update(v for v in commit_count_repos[row['Repo']].values() if v > 1)

# Sort the counts for plotting
spdx_x, spdx_y = zip(*sorted(spdx_counter.items()))
cyclonedx_x, cyclonedx_y = zip(*sorted(cyclonedx_counter.items()))

# Create the line plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=spdx_x,
    y=spdx_y,
    mode='lines+markers',
    name='SPDX',
    line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=cyclonedx_x,
    y=cyclonedx_y,
    mode='lines+markers',
    name='CycloneDx',
    line=dict(color='red')
))

# Update layout
fig.update_layout(
    # title='Commit Count Distribution for SPDX and CycloneDx',
    xaxis=dict(title='Number of Commits per Contributor (log scale)', type='log'),
    yaxis_title='Frequency',
    legend_title='Format',
    margin=dict(l=0, r=0, t=0, b=0),
    legend=dict(
        title='Format',
        x=0.8,
        y=0.8,
        bgcolor='rgba(255, 255, 255, 0.5)',
        bordercolor='black',
        borderwidth=1
    )
)

pio.write_image(fig, 'commit_count_distribution.pdf', format='pdf')

# Show the plot
fig.show()